## End-to-End ML Platform for Financial Fraud / Risk Scoring


# Import libraries

In [1]:
import pandas as pd
import numpy as np
import os, kagglehub
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score
import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

# Load dataset

In [2]:
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
data = pd.read_csv(os.path.join(path, "creditcard.csv"))

In [3]:
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


# Performing Exploratory Data Analysis(EDA) to understand structure and quality of data

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

# Checking the value count of class that refers as 1 as fraud transaction and 0 as Legit transaction

In [5]:
data["Class"].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

This data is highly unbalanced,
seperating data to do analysis

In [6]:
legit = data[data.Class == 0]
fraud = data[data.Class == 1]

In [7]:
print(legit.shape)
print(fraud.shape)

(284315, 31)
(492, 31)


In [8]:
legit.Amount.describe()

count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64

In [9]:
fraud.Amount.describe()

count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64

In [10]:
data.groupby("Class").mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,94838.202258,0.008258,-0.006271,0.012171,-0.007860,0.005453,0.002419,0.009637,-0.000987,0.004467,...,-0.000644,-0.001235,-0.000024,0.000070,0.000182,-0.000072,-0.000089,-0.000295,-0.000131,88.291022
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


As we can see there is huge difference between the mean of legit data and fraud data so doing unsampling by distributing dataset randomly into equal number for both the transaction

In [11]:
legit_sample = legit.sample(n=492)

Now Concatenate both the data into one dataset by using concat from pandas

In [12]:
new_data = pd.concat([legit_sample, fraud], axis=0) # axis = 0(row) and axis = 1(column)

In [13]:
new_data.head(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
93840,64608.0,-1.644137,-1.865167,0.972148,-0.723707,0.120402,5.485330,0.487287,1.311577,1.096663,...,0.354340,0.301079,0.857407,1.033622,0.858371,0.679247,-0.180307,0.017258,529.50,0
181364,124945.0,0.068755,0.911460,-0.365482,-0.433824,1.128152,-0.864257,1.226360,-0.588397,0.294001,...,0.214399,1.067990,-0.305224,-0.417759,-0.405459,-0.219312,-0.225883,-0.251231,4.33,0
45016,42171.0,-4.095274,3.332852,0.116662,-2.316591,-0.670862,-0.843148,0.816841,-0.259257,4.346171,...,-0.905507,-0.504155,0.051474,-0.030977,0.243983,0.584355,-0.001861,-1.173067,0.77,0
71479,54313.0,1.379941,-0.780671,1.204493,-0.704580,-1.609236,-0.323514,-1.342601,0.069858,-0.128903,...,0.409317,1.144324,-0.122904,0.077894,0.358877,-0.020055,0.056766,0.029123,9.99,0
12332,21584.0,-0.243917,0.930242,1.450462,1.520524,0.211802,0.005958,0.411864,-0.044222,0.327919,...,0.102460,0.523649,-0.135209,-0.029219,-0.268339,-0.167322,0.146795,0.145516,39.10,0


In [14]:
new_data["Class"].value_counts()

Class
0    492
1    492
Name: count, dtype: int64

In [15]:
new_data.groupby('Class').mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,94253.146341,-0.053213,-0.127794,0.002004,0.025064,0.002343,0.016853,-0.018823,-0.059005,0.053226,...,0.038262,-0.000115,0.026132,-0.012628,0.003571,0.013134,-0.036989,-0.007081,-0.008292,109.084289
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


Split the dataset into X and Y for classification

In [16]:
X = new_data.drop(columns="Class", axis=1)
Y = new_data["Class"]

In [17]:
X

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
93840,64608.0,-1.644137,-1.865167,0.972148,-0.723707,0.120402,5.485330,0.487287,1.311577,1.096663,...,1.212085,0.354340,0.301079,0.857407,1.033622,0.858371,0.679247,-0.180307,0.017258,529.50
181364,124945.0,0.068755,0.911460,-0.365482,-0.433824,1.128152,-0.864257,1.226360,-0.588397,0.294001,...,0.115627,0.214399,1.067990,-0.305224,-0.417759,-0.405459,-0.219312,-0.225883,-0.251231,4.33
45016,42171.0,-4.095274,3.332852,0.116662,-2.316591,-0.670862,-0.843148,0.816841,-0.259257,4.346171,...,2.228293,-0.905507,-0.504155,0.051474,-0.030977,0.243983,0.584355,-0.001861,-1.173067,0.77
71479,54313.0,1.379941,-0.780671,1.204493,-0.704580,-1.609236,-0.323514,-1.342601,0.069858,-0.128903,...,0.100625,0.409317,1.144324,-0.122904,0.077894,0.358877,-0.020055,0.056766,0.029123,9.99
12332,21584.0,-0.243917,0.930242,1.450462,1.520524,0.211802,0.005958,0.411864,-0.044222,0.327919,...,0.200600,0.102460,0.523649,-0.135209,-0.029219,-0.268339,-0.167322,0.146795,0.145516,39.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279863,169142.0,-1.927883,1.125653,-4.518331,1.749293,-1.566487,-2.010494,-0.882850,0.697211,-2.064945,...,1.252967,0.778584,-0.319189,0.639419,-0.294885,0.537503,0.788395,0.292680,0.147968,390.00
280143,169347.0,1.378559,1.289381,-5.004247,1.411850,0.442581,-1.326536,-1.413170,0.248525,-1.127396,...,0.226138,0.370612,0.028234,-0.145640,-0.081049,0.521875,0.739467,0.389152,0.186637,0.76
280149,169351.0,-0.676143,1.126366,-2.213700,0.468308,-1.120541,-0.003346,-2.234739,1.210158,-0.652250,...,0.247968,0.751826,0.834108,0.190944,0.032070,-0.739695,0.471111,0.385107,0.194361,77.89
281144,169966.0,-3.113832,0.585864,-5.399730,1.817092,-0.840618,-2.943548,-2.208002,1.058733,-1.632333,...,0.306271,0.583276,-0.269209,-0.456108,-0.183659,-0.328168,0.606116,0.884876,-0.253700,245.00


In [18]:
Y

93840     0
181364    0
45016     0
71479     0
12332     0
         ..
279863    1
280143    1
280149    1
281144    1
281674    1
Name: Class, Length: 984, dtype: int64

Using train-test split from SKlearn for train and test

In [19]:
X_train, X_test , y_train, y_test = train_test_split(X, Y, test_size=0.2)

# Model Training

Lets create a pipeline for three different model and compare them.

In [20]:
Models = {
    "RandomForest": RandomForestClassifier(n_estimators= 200, random_state= 42),
    "XGBoost": xgb.XGBClassifier(eval_metric = ["logloss", "auc"] ,n_estimators= 200, random_state= 42),
    "LightGBM": lgb.LGBMClassifier(n_estimators= 200, random_state= 42, verbosity = -1)
}

In [21]:
Results = {}

for name, model in Models.items():
  # train model
    model.fit(X_train, y_train)

    # Get predicted probabilities
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob = model.predict_proba(X_test)[:, 1]

    # Convert probabilities to binary predictions
    y_train_pred = (y_train_prob >= 0.5).astype(int)
    y_test_pred = (y_test_prob >= 0.5).astype(int)

    # Metrics
    roc_train = float(roc_auc_score(y_train, y_train_prob))
    roc_test = float(roc_auc_score(y_test, y_test_prob))
    precision = float(precision_score(y_test, y_test_pred))
    recall = float(recall_score(y_test, y_test_pred))
    accuracy = float(accuracy_score(y_test, y_test_pred))

  # check overfit/ underfit
    if roc_train - roc_test > 0.1:
        fit_status = "Overfit"
    elif roc_train < 0.6 and roc_test < 0.6:
        fit_status = "Underfit"
    else:
        fit_status = "Good fit"

    Results[name] = {
        "model": model,
         "metrics": {
            "roc_train": roc_train,
            "roc_test": roc_test,
            "precision": precision,
            "recall": recall,
            "accuracy": accuracy
        },
        "fit_status": fit_status
    }

In [22]:
Result_df = pd.DataFrame({
    model_name: {**metrics_info['metrics'], 'fit_status': metrics_info['fit_status']}
    for model_name, metrics_info in Results.items()
}).T

# Ensure numeric columns are float
numeric_cols = ['roc_train', 'roc_test', 'precision', 'recall', 'accuracy']
Result_df[numeric_cols] = Result_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Show comparison table
print("\n== Model Comparison ==")
print(Result_df)


== Model Comparison ==
              roc_train  roc_test  precision    recall  accuracy fit_status
RandomForest        1.0  0.981907   0.944444  0.913978  0.934010   Good fit
XGBoost             1.0  0.981907   0.956522  0.946237  0.954315   Good fit
LightGBM            1.0  0.984491   0.966292  0.924731  0.949239   Good fit


In [23]:
best_model = Result_df['roc_test'].idxmax()
best_model_score = Result_df.loc[best_model, 'roc_test']

print(f"\nBest Model: {best_model} with ROC Test Score: {best_model_score:.4f}")


Best Model: LightGBM with ROC Test Score: 0.9845


# Save the model using joblib

In [24]:
joblib.dump(best_model, 'C:/Users/Ronak/OneDrive/Desktop/Credit/Fraud_scoring.pkl')

['C:/Users/Ronak/OneDrive/Desktop/Credit/Fraud_scoring.pkl']

## Performing MLflow Experiment

In [25]:
Results

{'RandomForest': {'model': RandomForestClassifier(n_estimators=200, random_state=42),
  'metrics': {'roc_train': 1.0,
   'roc_test': 0.9819065343258893,
   'precision': 0.9444444444444444,
   'recall': 0.9139784946236559,
   'accuracy': 0.934010152284264},
  'fit_status': 'Good fit'},
 'XGBoost': {'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=None, early_stopping_rounds=None,
                enable_categorical=False, eval_metric=['logloss', 'auc'],
                feature_types=None, feature_weights=None, gamma=None,
                grow_policy=None, importance_type=None,
                interaction_constraints=None, learning_rate=None, max_bin=None,
                max_cat_threshold=None, max_cat_to_onehot=None,
                max_delta_step=None, max_depth=None, max_leaves=None,
                min_child_weight=None, missing=nan, monotone_constraints=N

In [26]:
print(type(Results))
print(Results)

<class 'dict'>
{'RandomForest': {'model': RandomForestClassifier(n_estimators=200, random_state=42), 'metrics': {'roc_train': 1.0, 'roc_test': 0.9819065343258893, 'precision': 0.9444444444444444, 'recall': 0.9139784946236559, 'accuracy': 0.934010152284264}, 'fit_status': 'Good fit'}, 'XGBoost': {'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=['logloss', 'auc'],
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              m

In [27]:
mlflow.end_run()  

In [28]:
mlflow.set_tracking_uri(uri = "http://127.0.0.1:5000/")
mlflow.set_experiment("Creditcardfraud_Models")

for model_name, data in Results.items():

    with mlflow.start_run(run_name=model_name):

        # Log model
        mlflow.sklearn.log_model(
            sk_model=data["model"],
            artifact_path="model"
        )

        # Log metrics
        for metric_name, metric_value in data["metrics"].items():
            mlflow.log_metric(metric_name, metric_value)

        # Log tags
        mlflow.set_tag("fit_status", data["fit_status"])
        mlflow.set_tag("model_name", model_name)
print("DONE")

2026/01/27 21:55:53 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/01/27 21:55:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instea

🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/7963b8b32474479b9e995d8c3a61e9b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/01/27 21:56:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/2/runs/0f65b1b7d5cf4a05a80a42810c907d1e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/2/runs/628f996c410a4b09b78e3e43dcb17ac6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
DONE


In [29]:
print(mlflow.get_tracking_uri())

http://127.0.0.1:5000/


## Register the model in MLflow

In [30]:
model_name = 'LightGBM'
run_id = input("Enter Run ID")
model_uri = f"runs:/{run_id}/model"

final = mlflow.register_model(
    model_uri, 
    model_name)

Enter Run ID628f996c410a4b09b78e3e43dcb17ac6


Registered model 'LightGBM' already exists. Creating a new version of this model...
2026/01/27 21:56:36 WARNING mlflow.tracking._model_registry.fluent: Run with id 628f996c410a4b09b78e3e43dcb17ac6 has no artifacts at artifact path 'model', registering model based on models:/m-ab1f710972604fa984e1050ff573529d instead
2026/01/27 21:56:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM, version 1
Created version '1' of model 'LightGBM'.


## Load the model

In [34]:
model_name = 'LightGBM'
model_version = 1
model_uri = f"models:/{model_name}@challenger"  # models:/<YOUR_MODEL_NAME>/<YOUR_MODEL_VERSION>

loaded_model = mlflow.sklearn.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([1, 0, 0, 1], dtype=int64)

In [37]:
dev_model_uri = f"models:/{model_name}@challenger"
prod_model = "credit-card-fraud-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri = dev_model_uri , dst_name = prod_model)

Successfully registered model 'credit-card-fraud-prod'.
Copied version '1' of model 'LightGBM' to version '1' of model 'credit-card-fraud-prod'.


<ModelVersion: aliases=[], creation_timestamp=1769570102036, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1769570102036, metrics=None, model_id=None, name='credit-card-fraud-prod', params=None, run_id='628f996c410a4b09b78e3e43dcb17ac6', run_link='', source='models:/LightGBM/1', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [38]:
model_uri = f"models:/{prod_model}@champion"  # models:/<YOUR_MODEL_NAME>/<YOUR_MODEL_VERSION>

loaded_model = mlflow.sklearn.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([1, 0, 0, 1], dtype=int64)